### pd.Series

In [3]:
class Series:
    def __init__(self, data=None, index=None, dtype=None, name=None):
        if data is None:
            self._values = []
            auto_index = []

        elif isinstance(data, dict):
            auto_index = list(data.keys())
            self._values = list(data.values())

        elif isinstance(data, list):
            self._values = data[:]
            auto_index = list(range(len(data)))

        elif isinstance(data, Series):
            self._values = data._values[:]
            auto_index = data._index[:]

        else:
            self._values = [data]
            auto_index = [0]

        if index is not None:
            if len(index) != len(self._values):
                raise ValueError(
                    f"Length of values ({len(self._values)}) does not match "
                    f"length of index ({len(index)})"
                )
            self._index = list(index)
        else:
            self._index = auto_index

        self.name = name
        self.dtype = dtype

    # ── representation ──────────────────────────────────────────────
    def __repr__(self):
        if len(self._values) == 0:
            name_part = f", Name: {self.name}" if self.name else ""
            return f"Series([]{name_part}, dtype: {self.dtype or 'object'})"

        idx_width = max(len(str(i)) for i in self._index)
        lines = [f"{str(idx):<{idx_width}}    {val}"
                 for idx, val in zip(self._index, self._values)]

        if self.name:
            lines.append(f"Name: {self.name}, dtype: {self.dtype or 'object'}")
        else:
            lines.append(f"dtype: {self.dtype or 'object'}")

        return "\n".join(lines)

    def __str__(self):
        return self.__repr__()

    # ── length ───────────────────────────────────────────────────────
    def __len__(self):
        return len(self._values)

    # ── getitem / setitem ────────────────────────────────────────────
    def __getitem__(self, key):
        if isinstance(key, int) and key not in self._index:
            return self._values[key]
        if key in self._index:
            i = self._index.index(key)
            return self._values[i]
        raise KeyError(f"Key '{key}' not found in index")

    def __setitem__(self, key, value):
        if key in self._index:
            i = self._index.index(key)
            self._values[i] = value
        else:
            self._index.append(key)
            self._values.append(value)

    # ── properties ───────────────────────────────────────────────────
    @property
    def values(self):
        return self._values[:]

    @property
    def index(self):
        return self._index[:]

    @property
    def shape(self):
        return (len(self._values),)

    @property
    def size(self):
        return len(self._values)

    # ── head / tail ──────────────────────────────────────────────────
    def head(self, n=5):
        return Series(self._values[:n], index=self._index[:n], name=self.name)

    def tail(self, n=5):
        return Series(self._values[-n:], index=self._index[-n:], name=self.name)

    # ── basic stats ──────────────────────────────────────────────────
    def sum(self):
        return sum(self._values)

    def mean(self):
        if len(self._values) == 0:
            return float('nan')
        return sum(self._values) / len(self._values)

    def min(self):
        return min(self._values)

    def max(self):
        return max(self._values)

    def count(self):
        return sum(1 for v in self._values if v is not None)

    def std(self):
        import math
        n = len(self._values)
        if n < 2:
            return float('nan')
        m = self.mean()
        return math.sqrt(sum((v - m) ** 2 for v in self._values) / (n - 1))

    def var(self):
        n = len(self._values)
        if n < 2:
            return float('nan')
        m = self.mean()
        return sum((v - m) ** 2 for v in self._values) / (n - 1)

    def median(self):
        sorted_vals = sorted(self._values)
        n = len(sorted_vals)
        if n == 0:
            return float('nan')
        mid = n // 2
        if n % 2 == 0:
            return (sorted_vals[mid - 1] + sorted_vals[mid]) / 2
        return float(sorted_vals[mid])

    def describe(self):
        return Series(
            [self.count(), self.mean(), self.std(),
             self.min(), self.median(), self.max()],
            index=['count', 'mean', 'std', 'min', '50%', 'max'],
            name=self.name
        )

    # ── arithmetic ───────────────────────────────────────────────────
    def _align(self, other):
        if isinstance(other, Series):
            return other._values
        return [other] * len(self._values)

    def __add__(self, other):
        vals = self._align(other)
        return Series([a + b for a, b in zip(self._values, vals)],
                      index=self._index[:], name=self.name)

    def __sub__(self, other):
        vals = self._align(other)
        return Series([a - b for a, b in zip(self._values, vals)],
                      index=self._index[:], name=self.name)

    def __mul__(self, other):
        vals = self._align(other)
        return Series([a * b for a, b in zip(self._values, vals)],
                      index=self._index[:], name=self.name)

    def __truediv__(self, other):
        vals = self._align(other)
        return Series([a / b for a, b in zip(self._values, vals)],
                      index=self._index[:], name=self.name)

    # ── comparison → boolean series ──────────────────────────────────
    def __gt__(self, other):
        vals = self._align(other)
        return Series([a > b for a, b in zip(self._values, vals)],
                      index=self._index[:])

    def __lt__(self, other):
        vals = self._align(other)
        return Series([a < b for a, b in zip(self._values, vals)],
                      index=self._index[:])

    def __eq__(self, other):
        vals = self._align(other)
        return Series([a == b for a, b in zip(self._values, vals)],
                      index=self._index[:])

    # ── boolean indexing ─────────────────────────────────────────────
    def __and__(self, other):
        return Series([a and b for a, b in zip(self._values, other._values)],
                      index=self._index[:])

    def __or__(self, other):
        return Series([a or b for a, b in zip(self._values, other._values)],
                      index=self._index[:])

    # ── apply / map ──────────────────────────────────────────────────
    def apply(self, func):
        return Series([func(v) for v in self._values],
                      index=self._index[:], name=self.name)

    def map(self, func_or_dict):
        if isinstance(func_or_dict, dict):
            return Series([func_or_dict.get(v, None) for v in self._values],
                          index=self._index[:], name=self.name)
        return Series([func_or_dict(v) for v in self._values],
                      index=self._index[:], name=self.name)

    # ── isnull / fillna / dropna ─────────────────────────────────────
    def isnull(self):
        return Series([v is None for v in self._values],
                      index=self._index[:])

    def notnull(self):
        return Series([v is not None for v in self._values],
                      index=self._index[:])

    def fillna(self, value):
        return Series([value if v is None else v for v in self._values],
                      index=self._index[:], name=self.name)

    def dropna(self):
        pairs = [(i, v) for i, v in zip(self._index, self._values) if v is not None]
        if not pairs:
            return Series([], name=self.name)
        idx, vals = zip(*pairs)
        return Series(list(vals), index=list(idx), name=self.name)

    # ── value_counts ─────────────────────────────────────────────────
    def value_counts(self):
        counts = {}
        for v in self._values:
            counts[v] = counts.get(v, 0) + 1
        sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
        return Series([c for _, c in sorted_counts],
                      index=[k for k, _ in sorted_counts])

    # ── sort ─────────────────────────────────────────────────────────
    def sort_values(self, ascending=True):
        pairs = sorted(zip(self._values, self._index), reverse=not ascending)
        vals, idx = zip(*pairs) if pairs else ([], [])
        return Series(list(vals), index=list(idx), name=self.name)

    def sort_index(self, ascending=True):
        pairs = sorted(zip(self._index, self._values), reverse=not ascending)
        idx, vals = zip(*pairs) if pairs else ([], [])
        return Series(list(vals), index=list(idx), name=self.name)

    # ── iloc / loc ───────────────────────────────────────────────────
    @property
    def iloc(self):
        return _iLocIndexer(self)

    @property
    def loc(self):
        return _LocIndexer(self)


# ── indexers ─────────────────────────────────────────────────────────
class _iLocIndexer:
    def __init__(self, series):
        self._s = series

    def __getitem__(self, key):
        if isinstance(key, int):
            return self._s._values[key]
        if isinstance(key, slice):
            return Series(self._s._values[key],
                          index=self._s._index[key], name=self._s.name)
        if isinstance(key, list):
            return Series([self._s._values[i] for i in key],
                          index=[self._s._index[i] for i in key],
                          name=self._s.name)
        raise TypeError(f"Cannot index with type {type(key)}")


class _LocIndexer:
    def __init__(self, series):
        self._s = series

    def __getitem__(self, key):
        if isinstance(key, list):
            result_idx, result_vals = [], []
            for k in key:
                if k not in self._s._index:
                    raise KeyError(k)
                i = self._s._index.index(k)
                result_idx.append(k)
                result_vals.append(self._s._values[i])
            return Series(result_vals, index=result_idx, name=self._s.name)
        if key not in self._s._index:
            raise KeyError(key)
        i = self._s._index.index(key)
        return self._s._values[i]

### test cases 

In [5]:
# normal
s1 = Series([10, 20, 30])
print(s1._values, s1._index)

# custom index
s2 = Series([10, 20, 30], index=['a', 'b', 'c'])
print(s2._index)

# from dict
s3 = Series({'x': 1, 'y': 2, 'z': 3})
print(s3._values, s3._index)

# scalar
s4 = Series(42)
print(s4._values, s4._index)

# None
s5 = Series()
print(s5._values, s5._index)

# with name
s6 = Series([1, 2, 3], name='scores')
print(s6.name)

# from another Series
s7 = Series(s2)
print(s7._values, s7._index)

# ERROR: index length mismatch
try:
    Series([1, 2, 3], index=['a', 'b'])
except ValueError as e:
    print("Caught:", e)

[10, 20, 30] [0, 1, 2]
['a', 'b', 'c']
[1, 2, 3] ['x', 'y', 'z']
[42] [0]
[] []
scores
[10, 20, 30] ['a', 'b', 'c']
Caught: Length of values (3) does not match length of index (2)


In [6]:
print(Series([10, 20, 30]))
print(Series([10, 20, 30], index=['a', 'b', 'c'], name='scores'))
print(Series([]))
print(Series({'x': 1, 'y': 2})) 

0    10
1    20
2    30
dtype: object
a    10
b    20
c    30
Name: scores, dtype: object
Series([], dtype: object)
x    1
y    2
dtype: object


In [7]:
print(len(Series([1, 2, 3])))       
print(len(Series([])))              
print(len(Series(42)))

3
0
1


In [8]:
s = Series([10, 20, 30], index=['a', 'b', 'c'])
print(s['a'])       
print(s['c'])       
s['b'] = 99
print(s['b'])       
s['d'] = 55         
print(s._index)     

10
30
99
['a', 'b', 'c', 'd']


In [9]:
s = Series([1, 2, 3, 4, 5, 6, 7])
print(s.head())
print(s.head(3))
print(s.tail())
print(s.tail(2)) 

0    1
1    2
2    3
3    4
4    5
dtype: object
0    1
1    2
2    3
dtype: object
2    3
3    4
4    5
5    6
6    7
dtype: object
5    6
6    7
dtype: object


In [10]:
print(Series([1, 2, 3, 4, 5]).describe())

count    5
mean     3.0
std      1.5811388300841898
min      1
50%      3.0
max      5
dtype: object


In [11]:
s = Series([1, 2, 3, 4, 5])
print(s.sum())     
print(s.mean())    
print(s.min())      # 1
print(s.max())      # 5
print(s.std())      # 1.5811...
print(s.var())      # 2.5
print(s.median())   # 3.0

# edge: empty
print(Series([]).mean())   
print(Series([]).median()) 

# edge: single value
print(Series([42]).std())  

15
3.0
1
5
1.5811388300841898
2.5
3.0
nan
nan
nan


In [12]:
s = Series([1, 2, 3])
print(s + 10)
print(s - 1)
print(s * 2)
print(s / 2)
print(s + Series([10, 20, 30])) 

0    11
1    12
2    13
dtype: object
0    0
1    1
2    2
dtype: object
0    2
1    4
2    6
dtype: object
0    0.5
1    1.0
2    1.5
dtype: object
0    11
1    22
2    33
dtype: object


In [13]:
s = Series([1, 2, 3, 4, 5])
print(s > 3)
print(s < 2)
print(s == 3) 

0    False
1    False
2    False
3    True
4    True
dtype: object
0    True
1    False
2    False
3    False
4    False
dtype: object
0    False
1    False
2    True
3    False
4    False
dtype: object


In [14]:
s = Series([1, None, 3, None, 5])
print(s.isnull())
print(s.notnull())
print(s.fillna(0))
print(s.dropna()) 

0    False
1    True
2    False
3    True
4    False
dtype: object
0    True
1    False
2    True
3    False
4    True
dtype: object
0    1
1    0
2    3
3    0
4    5
dtype: object
0    1
2    3
4    5
dtype: object


In [15]:
s = Series([1, 2, 3, 4])
print(s.apply(lambda x: x ** 2))
print(s.map({1: 'one', 2: 'two', 3: 'three', 4: 'four'}))
print(s.map(lambda x: x * 10)) 

0    1
1    4
2    9
3    16
dtype: object
0    one
1    two
2    three
3    four
dtype: object
0    10
1    20
2    30
3    40
dtype: object


In [16]:
print(Series(['a', 'b', 'a', 'c', 'b', 'a']).value_counts()) 

a    3
b    2
c    1
dtype: object


In [17]:
s = Series([3, 1, 4, 1, 5], index=['e', 'a', 'c', 'b', 'd'])
print(s.sort_values())
print(s.sort_values(ascending=False))
print(s.sort_index())
print(s.sort_index(ascending=False)) 

a    1
b    1
e    3
c    4
d    5
dtype: object
d    5
c    4
e    3
b    1
a    1
dtype: object
a    1
b    1
c    4
d    5
e    3
dtype: object
e    3
d    5
c    4
b    1
a    1
dtype: object


In [18]:
s = Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print(s.iloc[0])           # 10
print(s.iloc[-1])          # 40
print(s.iloc[1:3])         # b→20, c→30
print(s.iloc[[0, 2]])      # a→10, c→30

print(s.loc['a'])          # 10
print(s.loc[['a', 'c']])   # a→10, c→30

# ERROR: loc key not found
try:
    s.loc['z']
except KeyError as e:
    print("Caught:", e)

# ERROR: iloc out of range
try:
    s.iloc[100]
except IndexError as e:
    print("Caught:", e) 

10
40
b    20
c    30
dtype: object
a    10
c    30
dtype: object
10
a    10
c    30
dtype: object
Caught: 'z'
Caught: list index out of range
